# OpenTargetsAdapter Example Notebook

This notebook demonstrates how to use the `OpenTargetsAdapter` to search for and retrieve detailed information about biological targets and diseases from the OpenTargets Platform.

## Features

- Search for targets and diseases by keyword
- Retrieve detailed concept information
- Access gene-level data (biotype, ApprovedSymbol)
- Access disease-level data (definitions, names)
- Navigate the OpenTargets GraphQL API

## Setup

The OpenTargets API is free to use without authentication.

In [16]:
from knowledge_lookup.adapters.opentargets_adapter import OpenTargetsAdapter
from knowledge_lookup.models import LookupConfig

# Initialize the adapter with a config
config = LookupConfig()
adapter = OpenTargetsAdapter(config)

# Verify the adapter is available
print(f"Adapter available: {adapter.is_available()}")
print(f"Base URL: {adapter.base_url}")

Adapter available: True
Base URL: https://api.platform.opentargets.org/api/v4/graphql


## Section 1: Search for Targets

Search for biological targets (genes/proteins) by keyword. The search returns a list of `UnifiedConcept` objects containing target information.

In [17]:
# Search for targets related to cancer
results = await adapter.search_concepts('cancer', limit=5)

print(f"Found {len(results)} results for 'cancer':\n")

for i, concept in enumerate(results, 1):
    print(f"{i}. ID: {concept.primary_id}")
    print(f"   Label: {concept.primary_label}")
    print(f"   Type: {concept.concept_type}")
    if concept.identifiers:
        print(f"   URL: {concept.identifiers[0].url}")
    print()

Found 5 results for 'cancer':

1. ID: MONDO_0004992
   Label: MONDO_0004992
   Type: ConceptType.GENE
   URL: https://platform.opentargets.org/target/MONDO_0004992

2. ID: Orphanet_145
   Label: Orphanet_145
   Type: ConceptType.GENE
   URL: https://platform.opentargets.org/target/Orphanet_145

3. ID: ENSG00000139618
   Label: ENSG00000139618
   Type: ConceptType.GENE
   URL: https://platform.opentargets.org/target/ENSG00000139618

4. ID: MONDO_0003582
   Label: MONDO_0003582
   Type: ConceptType.GENE
   URL: https://platform.opentargets.org/target/MONDO_0003582

5. ID: ENSG00000012048
   Label: ENSG00000012048
   Type: ConceptType.GENE
   URL: https://platform.opentargets.org/target/ENSG00000012048



In [18]:
# Search for a specific gene
results = await adapter.search_concepts('BRCA1', limit=3)

print(f"Search results for 'BRCA1':\n")
for concept in results:
    print(f"ID: {concept.primary_id}")
    print(f"Label: {concept.primary_label}")
    print(f"Type: {concept.concept_type}")
    print()

Search results for 'BRCA1':

ID: ENSG00000012048
Label: ENSG00000012048
Type: ConceptType.GENE

ID: MONDO_0700268
Label: MONDO_0700268
Type: ConceptType.GENE

ID: MONDO_0013685
Label: MONDO_0013685
Type: ConceptType.GENE



## Section 2: Search for Diseases

Search for diseases and phenotypes by keyword.

In [19]:
# Search for diabetes-related diseases
results = await adapter.search_concepts('diabetes', limit=5)

print(f"Found {len(results)} results for 'diabetes':\n")

for i, concept in enumerate(results, 1):
    print(f"{i}. ID: {concept.primary_id}")
    print(f"   Label: {concept.primary_label}")
    print(f"   Type: {concept.concept_type}")
    print()

Found 5 results for 'diabetes':

1. ID: EFO_0000400
   Label: EFO_0000400
   Type: ConceptType.GENE

2. ID: MONDO_0005148
   Label: MONDO_0005148
   Type: ConceptType.GENE

3. ID: EFO_1001511
   Label: EFO_1001511
   Type: ConceptType.GENE

4. ID: EFO_0004593
   Label: EFO_0004593
   Type: ConceptType.GENE

5. ID: MONDO_0007450
   Label: MONDO_0007450
   Type: ConceptType.GENE



In [20]:
# Search for Alzheimer's disease
results = await adapter.search_concepts('Alzheimer', limit=5)

print(f"Found {len(results)} results for 'Alzheimer':\n")
for i, concept in enumerate(results, 1):
    print(f"{i}. {concept.primary_label} ({concept.primary_id})")

Found 5 results for 'Alzheimer':

1. MONDO_0004975 (MONDO_0004975)
2. MONDO_0100087 (MONDO_0100087)
3. MONDO_0015140 (MONDO_0015140)
4. EFO_1001870 (EFO_1001870)
5. EFO_0022957 (EFO_0022957)


## Section 3: Get Detailed Concept Information

Retrieve detailed information for a specific target or disease using its ID.

In [21]:
# First, search for a target to get an ID
search_results = await adapter.search_concepts('BRCA1', limit=1)

if search_results:
    target_id = search_results[0].primary_id
    print(f"Searching for: {search_results[0].primary_label}")
    print(f"Target ID: {target_id}")
    print()

    # Get detailed information
    details = await adapter.get_concept_details(target_id)

    if details:
        print("=== Detailed Concept Information ===")
        print(f"Primary ID: {details.primary_id}")
        print(f"Primary Label: {details.primary_label}")
        print(f"Concept Type: {details.concept_type}")
        print(f"Definition: {details.definitions[0] if details.definitions else 'N/A'}")
        print()
        print("Identifiers:")
        for identifier in details.identifiers:
            print(f"  - Source: {identifier.source}")
            print(f"    URL: {identifier.url}")
        print()
        print(f"Synonyms: {details.synonyms[:5] if details.synonyms else []}")
    else:
        print("No details found for this target.")

Searching for: ENSG00000012048
Target ID: ENSG00000012048

=== Detailed Concept Information ===
Primary ID: ENSG00000012048
Primary Label: BRCA1
Concept Type: ConceptType.GENE
Definition: Biotype: protein_coding

Identifiers:
  - Source: KnowledgeSource.OPENTARGETS
    URL: https://platform.opentargets.org/target/ENSG00000012048

Synonyms: ['BRCA1']


## Section 4: Search by Specific Entity Type

You can search for either targets or diseases specifically by modifying the search query.

In [22]:
# Search for a specific Ensembl gene ID
gene_id = "ENSG00000139618"  # BRCA2 gene

# Search for the exact gene
results = await adapter.search_concepts(gene_id, limit=5)

print(f"Search results for {gene_id}:\n")

for concept in results:
    print(f"ID: {concept.primary_id}")
    print(f"Label: {concept.primary_label}")
    print(f"Type: {concept.concept_type}")
    print()

Search results for ENSG00000139618:

ID: ENSG00000139618
Label: ENSG00000139618
Type: ConceptType.GENE



## Section 5: Working with Search Results

Explore the structure of search results and extract useful information.

In [23]:
# Explore result structure
results = await adapter.search_concepts('inflammation', limit=3)

print(f"Exploring {len(results)} results for 'inflammation':\n")

for concept in results:
    # Print the full concept structure
    print(f"=== {concept.primary_label} ===")
    print(f"  Primary ID: {concept.primary_id}")
    print(f"  Concept Type: {concept.concept_type}")
    print(f"  Definitions: {concept.definitions}")
    print(f"  Synonyms: {concept.synonyms[:5] if concept.synonyms else []}")
    print(f"  Confidence Score: {concept.confidence_score}")
    print()

Exploring 3 results for 'inflammation':

=== MP_0001845 ===
  Primary ID: MP_0001845
  Concept Type: ConceptType.GENE
  Definitions: []
  Synonyms: []
  Confidence Score: 0.9

=== GO_0006954 ===
  Primary ID: GO_0006954
  Concept Type: ConceptType.GENE
  Definitions: []
  Synonyms: []
  Confidence Score: 0.9

=== MONDO_0002614 ===
  Primary ID: MONDO_0002614
  Concept Type: ConceptType.GENE
  Definitions: []
  Synonyms: []
  Confidence Score: 0.9



## Section 6: Working with Multiple Search Queries

Perform batch searches and compare results.

In [24]:
# Batch search for multiple terms
search_terms = ['cancer', 'diabetes', 'Alzheimer']
all_results = {}

for term in search_terms:
    results = await adapter.search_concepts(term, limit=3)
    all_results[term] = results
    print(f"'{term}': Found {len(results)} results")

print("\n--- Sample Results ---")
for term, results in all_results.items():
    if results:
        print(f"\n{term}:")
        for r in results[:2]:
            print(f"  - {r.primary_label} ({r.primary_id})")

'cancer': Found 3 results
'diabetes': Found 3 results
'Alzheimer': Found 3 results

--- Sample Results ---

cancer:
  - MONDO_0004992 (MONDO_0004992)
  - Orphanet_145 (Orphanet_145)

diabetes:
  - EFO_0000400 (EFO_0000400)
  - MONDO_0005148 (MONDO_0005148)

Alzheimer:
  - MONDO_0004975 (MONDO_0004975)
  - MONDO_0100087 (MONDO_0100087)


## Section 7: Limit and Pagination

Control the number of results returned with the limit parameter.

In [25]:
# Test different limit values
limits = [2, 5, 10]

for limit in limits:
    results = await adapter.search_concepts('kinase', limit=limit)
    print(f"Limit={limit}: Returned {len(results)} results")

print("\n--- First 3 results for 'kinase': ---")
results = await adapter.search_concepts('kinase', limit=3)
for i, concept in enumerate(results, 1):
    print(f"{i}. {concept.primary_label} - {concept.primary_id}")

Limit=2: Returned 2 results
Limit=5: Returned 5 results
Limit=10: Returned 10 results

--- First 3 results for 'kinase': ---
1. ENSG00000122025 - ENSG00000122025
2. ENSG00000128052 - ENSG00000128052
3. ENSG00000096968 - ENSG00000096968


## Section 8: Error Handling

Handle potential API errors gracefully.

In [26]:
# Handle potential errors
# Try to get details for a non-existent ID
non_existent_id = "ENSG00000999999"  # This gene doesn't exist

print(f"Trying to get details for: {non_existent_id}")

# First search
search_results = await adapter.search_concepts(non_existent_id, limit=1)
print(f"Search results: {len(search_results)}")

# Try to get details
if search_results:
    details = await adapter.get_concept_details(search_results[0].primary_id)
    print(f"Details found: {details is not None}")
else:
    print("No search results - cannot get details")

Trying to get details for: ENSG00000999999
Search results: 0
No search results - cannot get details


## Section 9: Comparing Targets vs Diseases

Demonstrate the difference between target and disease searches.

In [27]:
# Compare target and disease searches
print("=== Target Search (EGFR) ===")
target_results = await adapter.search_concepts('EGFR', limit=3)
for concept in target_results:
    print(f"Type: {concept.concept_type} - {concept.primary_label}")

print("\n=== Disease Search (EGFR) ===")
disease_results = await adapter.search_concepts('EGFR', limit=3)
for concept in disease_results:
    print(f"Type: {concept.concept_type} - {concept.primary_label}")

=== Target Search (EGFR) ===
Type: ConceptType.GENE - ENSG00000146648
Type: ConceptType.GENE - EFO_0022194
Type: ConceptType.GENE - MONDO_0014481

=== Disease Search (EGFR) ===
Type: ConceptType.GENE - ENSG00000146648
Type: ConceptType.GENE - EFO_0022194
Type: ConceptType.GENE - MONDO_0014481


## Conclusion

This notebook demonstrated the core functionality of the `OpenTargetsAdapter`:

- **Search**: Find targets and diseases by keyword
- **Details**: Retrieve comprehensive information about specific concepts
- **Results**: Work with structured `UnifiedConcept` objects
- **Limit**: Control result counts with the limit parameter

### Next Steps

- Explore more search terms related to your research area
- Combine OpenTargets data with other knowledge sources
- Integrate with your own analysis pipelines

### Resources

- [OpenTargets Platform](https://platform.opentargets.org/)
- [OpenTargets GraphQL API](https://api.platform.opentargets.org/api/v4/graphql)
- [OpenTargets Documentation](https://docs.opentargets.org/)